# 🔬 Erythemato-Squamous Disease Type Classifier

This notebook trains a **Random Forest** classifier on the UCI Dermatology dataset and provides an **interactive prediction tool** (powered by `ipywidgets`) so clinicians can enter a patient's feature values and instantly obtain a predicted disease type.

### Disease Classes
| Code | Disease |
|------|---------|
| 1 | Psoriasis |
| 2 | Seborrhoeic Dermatitis |
| 3 | Lichen Planus |
| 4 | Pityriasis Rosea |
| 5 | Chronic Dermatitis |
| 6 | Pityriasis Rubra Pilaris |

---
## Section 1 — Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Styling ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

# ── Disease label map ──────────────────────────────────────────────────────
CLASS_LABELS = {
    1: 'Psoriasis',
    2: 'Seborrhoeic Dermatitis',
    3: 'Lichen Planus',
    4: 'Pityriasis Rosea',
    5: 'Chronic Dermatitis',
    6: 'Pityriasis Rubra Pilaris',
}

# ── Feature groups for UI organisation ────────────────────────────────────
CLINICAL_FEATURES = [
    'erythema', 'scaling', 'definite_borders', 'itching',
    'koebner_phenomenon', 'polygonal_papules', 'follicular_papules',
    'oral_mucosal_involvement', 'knee_and_elbow_involvement',
    'scalp_involvement', 'family_history',
]

HISTOPATHOLOGICAL_FEATURES = [
    'melanin_incontinence', 'eosinophils_infiltrate', 'PNL_infiltrate',
    'fibrosis_papillary_dermis', 'exocytosis', 'acanthosis',
    'hyperkeratosis', 'parakeratosis', 'clubbing_rete_ridges',
    'elongation_rete_ridges', 'thinning_suprapapillary_epidermis',
    'spongiform_pustule', 'munro_microabcess', 'focal_hypergranulosis',
    'disappearance_granular_layer', 'vacuolisation_damage_basal_layer',
    'spongiosis', 'saw_tooth_appearance_retes', 'follicular_horn_plug',
    'perifollicular_parakeratosis', 'inflammatory_mononuclear_infiltrate',
    'band_like_infiltrate',
]

print('✅ Libraries loaded successfully.')

---
## Section 2 — Data Loading & Exploration

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────
df = pd.read_csv('dermatology_database.csv')

# Map integer class codes to human-readable labels for display purposes
df['disease'] = df['class'].map(CLASS_LABELS)

print(f'Dataset shape : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Features      : {df.shape[1] - 2}  (34 + class + disease label)')
print(f'Missing values: {df.isnull().sum().sum()}')
display(df.head(3))

In [ ]:
# ── Class distribution ─────────────────────────────────────────────────────
dist = df['disease'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
bars = axes[0].barh(
    dist.index, dist.values,
    color=sns.color_palette('muted', len(dist))
)
axes[0].bar_label(bars, padding=4, fontsize=10)
axes[0].set_xlabel('Number of Samples')
axes[0].set_title('Class Distribution — Erythemato-Squamous Diseases')
axes[0].invert_yaxis()

# Pie chart
axes[1].pie(
    dist.values,
    labels=dist.index,
    autopct='%1.1f%%',
    colors=sns.color_palette('muted', len(dist)),
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.4}
)
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.suptitle('Target Variable Distribution', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# ── Statistical summary ────────────────────────────────────────────────────
display(HTML('<h4>Statistical Summary of All Features</h4>'))
display(df.drop(columns=['class', 'disease']).describe().round(2))

In [ ]:
# ── Correlation heatmap (clinical features) ────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
corr = df[CLINICAL_FEATURES + HISTOPATHOLOGICAL_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.4, annot=False, cbar_kws={'shrink': 0.7}
)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Mean feature values by disease class ──────────────────────────────────
key_features = [
    'erythema', 'scaling', 'itching', 'acanthosis',
    'parakeratosis', 'band_like_infiltrate', 'spongiosis',
    'hyperkeratosis', 'exocytosis', 'koebner_phenomenon'
]

group_means = df.groupby('disease')[key_features].mean()

fig, ax = plt.subplots(figsize=(14, 6))
group_means.T.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', linewidth=0.5)
ax.set_title('Mean Feature Values by Disease Class (Key Features)', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Mean Score')
ax.legend(title='Disease', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
plt.tight_layout()
plt.show()

---
## Section 3 — Data Preprocessing

In [ ]:
FEATURE_COLS = [c for c in df.columns if c not in ('class', 'disease')]
TARGET_COL   = 'class'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

# Handle any missing values (the 'age' column may have '?' in some versions)
if X.isnull().sum().sum() > 0:
    X.fillna(X.median(), inplace=True)
    print('ℹ️  Missing values imputed with column medians.')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features used    : {len(FEATURE_COLS)}')

---
## Section 4 — Model Training

In [ ]:
# ── Train Random Forest ────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# ── Cross-validation ───────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy')

print(f'5-Fold CV Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'CV Fold Scores     : {np.round(cv_scores, 4)}')

---
## Section 5 — Model Evaluation

In [ ]:
# ── Test-set evaluation ────────────────────────────────────────────────────
y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

display(HTML(f'<h4>Test-Set Accuracy: <span style="color:#2d7dd2">{acc*100:.2f}%</span></h4>'))

target_names = [CLASS_LABELS[i] for i in sorted(CLASS_LABELS)]
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=30)
ax.set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importances ────────────────────────────────────────────────────
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': importances
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 12))
colors = ['#2d7dd2' if i >= len(feat_imp_df) - 10 else '#bcd4ec'
          for i in range(len(feat_imp_df))]
bars = ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color=colors, edgecolor='white')
ax.set_xlabel('Mean Decrease in Impurity (Gini Importance)')
ax.set_title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
top_patch   = mpatches.Patch(color='#2d7dd2',  label='Top-10 most important')
other_patch = mpatches.Patch(color='#bcd4ec', label='Other features')
ax.legend(handles=[top_patch, other_patch], loc='lower right')
plt.tight_layout()
plt.show()

---
## Section 6 — Interactive Prediction Dashboard

> **How to use:** Adjust the sliders to reflect the patient's clinical and histopathological scores (0 = absent, 1 = mild, 2 = moderate, 3 = severe), enter patient age, then click **🔍 Predict Disease**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  FRONTEND — Interactive Prediction UI (ipywidgets)
# ═══════════════════════════════════════════════════════════════════════════

# ── Helper: nice section header ────────────────────────────────────────────
def section_header(title, color='#2d7dd2'):
    return widgets.HTML(
        value=f'<div style="background:{color};color:white;padding:8px 14px;'
              f'border-radius:6px;font-weight:600;font-size:14px;'
              f'margin-bottom:8px">{title}</div>'
    )

# ── Slider factory ─────────────────────────────────────────────────────────
def make_slider(name, max_val=3):
    label = name.replace('_', ' ').title()
    slider = widgets.IntSlider(
        value=0, min=0, max=max_val, step=1,
        description='', continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='200px')
    )
    lbl = widgets.Label(
        value=label,
        layout=widgets.Layout(width='270px', margin='auto 0')
    )
    return widgets.HBox([lbl, slider]), slider

# ── Build slider groups ────────────────────────────────────────────────────
clinical_sliders   = {}
histopath_sliders  = {}
clinical_boxes     = []
histopath_boxes    = []

for feat in CLINICAL_FEATURES:
    max_v = 1 if feat == 'family_history' else 3
    box, slider = make_slider(feat, max_v)
    clinical_sliders[feat] = slider
    clinical_boxes.append(box)

for feat in HISTOPATHOLOGICAL_FEATURES:
    box, slider = make_slider(feat)
    histopath_sliders[feat] = slider
    histopath_boxes.append(box)

# Age widget
age_widget = widgets.BoundedIntText(
    value=35, min=1, max=110,
    description='Patient Age:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='250px')
)

# ── Action button & output ─────────────────────────────────────────────────
predict_btn = widgets.Button(
    description='🔍  Predict Disease',
    button_style='primary',
    tooltip='Run the trained model on the entered values',
    layout=widgets.Layout(width='200px', height='38px')
)
reset_btn = widgets.Button(
    description='↺  Reset',
    button_style='warning',
    layout=widgets.Layout(width='110px', height='38px')
)
output_area = widgets.Output(
    layout=widgets.Layout(border='1px solid #e5e7eb', padding='14px', border_radius='8px', margin_top='12px')
)

# ── Disease color map ──────────────────────────────────────────────────────
DISEASE_COLORS = {
    'Psoriasis':                '#e64d2e',
    'Seborrhoeic Dermatitis':   '#d4880a',
    'Lichen Planus':            '#7c5cd8',
    'Pityriasis Rosea':         '#2d9e6b',
    'Chronic Dermatitis':       '#2d7dd2',
    'Pityriasis Rubra Pilaris': '#b5490f',
}

# ── Brief disease descriptions ─────────────────────────────────────────────
DISEASE_INFO = {
    'Psoriasis':
        'A chronic autoimmune condition causing rapid skin cell buildup, '
        'resulting in scaling, red patches and itching.',
    'Seborrhoeic Dermatitis':
        'A common skin condition affecting the scalp, causing scaly patches, '
        'red skin and stubborn dandruff.',
    'Lichen Planus':
        'An inflammatory condition affecting skin, hair, nails and mucous membranes. '
        'Presents as purplish, itchy, flat-topped bumps.',
    'Pityriasis Rosea':
        'A generally mild rash that begins as a large scaly patch and spreads; '
        'often resolves on its own within weeks.',
    'Chronic Dermatitis':
        'Long-lasting skin inflammation causing redness, scaling, and intense itching, '
        'frequently associated with allergic responses.',
    'Pityriasis Rubra Pilaris':
        'A rare skin disorder causing persistent inflammation, redness and scaling; '
        'may involve large areas of the body.',
}

# ── Prediction callback ────────────────────────────────────────────────────
def on_predict(b):
    with output_area:
        clear_output(wait=True)
        values = {}
        for feat in CLINICAL_FEATURES:
            values[feat] = clinical_sliders[feat].value
        for feat in HISTOPATHOLOGICAL_FEATURES:
            values[feat] = histopath_sliders[feat].value
        values['age'] = age_widget.value

        row = pd.DataFrame([values])[FEATURE_COLS]
        pred_class = rf_model.predict(row)[0]
        pred_proba = rf_model.predict_proba(row)[0]

        pred_label = CLASS_LABELS[pred_class]
        color = DISEASE_COLORS.get(pred_label, '#2d7dd2')

        # ── Result card ────────────────────────────────────────────────────
        display(HTML(
            f'<div style="background:{color}18;border-left:5px solid {color};'
            f'padding:14px 18px;border-radius:6px;margin-bottom:10px">'
            f'<div style="font-size:20px;font-weight:700;color:{color}">'
            f'Predicted Disease: {pred_label}</div>'
            f'<div style="color:#444;margin-top:6px;font-size:13px">'
            f'{DISEASE_INFO[pred_label]}</div>'
            f'</div>'
        ))

        # ── Probability breakdown (horizontal bar chart) ───────────────────
        classes   = [CLASS_LABELS[i] for i in rf_model.classes_]
        probs     = pred_proba * 100
        bar_colors = [DISEASE_COLORS.get(c, '#888') for c in classes]

        fig, ax = plt.subplots(figsize=(8, 3.5))
        bars = ax.barh(classes, probs, color=bar_colors, edgecolor='white', height=0.55)
        ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=10)
        ax.set_xlim(0, 110)
        ax.set_xlabel('Probability (%)')
        ax.set_title('Prediction Confidence per Disease Class', fontsize=12, fontweight='bold')
        ax.invert_yaxis()
        # Highlight predicted bar
        for bar, cls in zip(ax.patches, classes):
            if cls == pred_label:
                bar.set_edgecolor('black')
                bar.set_linewidth(1.8)
        plt.tight_layout()
        plt.show()

        # ── Top features driving this prediction ───────────────────────────
        top_n = 8
        fi = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
        top_feats = fi.nlargest(top_n).sort_values()
        patient_vals = [values[f] for f in top_feats.index]

        fig2, ax2 = plt.subplots(figsize=(8, 3.5))
        palette = ['#2d7dd2' if v > 0 else '#bcd4ec' for v in patient_vals]
        ax2.barh(top_feats.index.str.replace('_', ' ').str.title(),
                 top_feats.values, color=palette, edgecolor='white')
        ax2.set_xlabel('Feature Importance')
        ax2.set_title(f'Top {top_n} Model Features (Patient values > 0 highlighted)', fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()


def on_reset(b):
    for s in clinical_sliders.values():
        s.value = 0
    for s in histopath_sliders.values():
        s.value = 0
    age_widget.value = 35
    with output_area:
        clear_output()


predict_btn.on_click(on_predict)
reset_btn.on_click(on_reset)

# ── Layout ─────────────────────────────────────────────────────────────────
title_banner = widgets.HTML(
    value='<div style="background:#1f2328;color:white;padding:14px 20px;'
          'border-radius:8px;font-size:18px;font-weight:700;margin-bottom:12px">'
          '🔬 Erythemato-Squamous Disease Classifier — Patient Assessment Tool</div>'
)

note_html = widgets.HTML(
    value='<div style="background:#fffbeb;border:1px solid #f59e0b;border-radius:6px;'
          'padding:10px 14px;font-size:12px;color:#92400e;margin-bottom:10px">'
          '<b>Score Guide:</b> 0 = absent &nbsp;|&nbsp; 1 = mild '
          '&nbsp;|&nbsp; 2 = moderate &nbsp;|&nbsp; 3 = severe/present. '
          'Family history: 0 = no, 1 = yes.</div>'
)

left_panel = widgets.VBox(
    [section_header('🩺 Clinical Features')] + clinical_boxes,
    layout=widgets.Layout(width='50%', padding='0 10px 0 0')
)
right_panel = widgets.VBox(
    [section_header('🔬 Histopathological Features', color='#7c5cd8')] + histopath_boxes,
    layout=widgets.Layout(width='50%', padding='0 0 0 10px')
)

feature_columns = widgets.HBox(
    [left_panel, right_panel],
    layout=widgets.Layout(margin='0 0 10px 0')
)

age_row = widgets.HBox(
    [section_header('👤 Patient Demographics', color='#2d9e6b'), age_widget],
    layout=widgets.Layout(align_items='center', margin='6px 0')
)

btn_row = widgets.HBox(
    [predict_btn, reset_btn],
    layout=widgets.Layout(margin='10px 0')
)

dashboard = widgets.VBox([
    title_banner,
    note_html,
    feature_columns,
    age_row,
    btn_row,
    output_area,
])

display(dashboard)

---
## Section 7 — Model Summary

In [ ]:
summary_data = {
    'Metric': [
        'Algorithm', 'Number of Trees', 'Training Samples',
        'Test Samples', 'Test Accuracy', '5-Fold CV Accuracy',
        'CV Std Dev', 'Number of Features'
    ],
    'Value': [
        'Random Forest', 200, len(X_train),
        len(X_test), f'{acc*100:.2f}%', f'{cv_scores.mean()*100:.2f}%',
        f'{cv_scores.std()*100:.2f}%', len(FEATURE_COLS)
    ]
}

summary_df = pd.DataFrame(summary_data)
display(HTML('<h4>📊 Model Performance Summary</h4>'))
display(summary_df.style.hide(axis='index').set_properties(**{
    'text-align': 'left', 'padding': '6px 16px'
}).set_table_styles([{
    'selector': 'th',
    'props': [('background-color', '#1f2328'), ('color', 'white'), ('padding', '8px 16px')]
}]))